# Лекция 1. Демонстрация: вопрос редактора проходит через систему

Курс «От интеллектуального поиска к интеллектуальным агентам» (ВМК МГУ). Блокнот показывается лектором на экране
в конце занятия; студенты повторяют его дома (часть 1 ДЗ). Зависимости: `pandas`, `rdflib`. Поиск BM25 реализован
в блокноте целиком, чтобы не зависеть от сторонних библиотек.

**Сквозной пример.** Редактор сборника собрал 8 статей и 14 рецензий. По «Положению о подготовке и рецензировании статей»
решение редколлегии принимается при не менее чем двух рецензиях и ответе авторов (п. 6.5), рецензент не из организации
авторов (6.2), рецензия сдаётся в 21 день (6.6). Вопрос редактора:

> **Какие статьи можно выносить на решение редколлегии, и на что опирается это требование?**

Шаги демонстрации:

0. Данные: журнал рецензий и корпус положений.
1. Модель без инструментов: запись ответа с тремя ошибками.
2. Поиск по словам (BM25): что находит по вопросу о зависимости.
3. Граф знаний: транзитивный путь `dependsOn+` от п. 6.5.
4. Проверка по данным: правила 6.1, 6.2, 6.5, 6.6 над журналом, с приведением шкалы.
5. Сборка ответа: найдено / проверено / не подтверждено / путь к источникам.
6. Что увидели: матрица «тип вопроса × канал».

Математический вариант того же скелета (корпус планиметрии) доступен переключателем `CORPUS = "A"` для шагов 2–3.

In [ ]:
import csv, math, re, collections, pathlib
import pandas as pd

DATA = pathlib.Path("data")
CORPUS = "Aprime"   # "Aprime" — положение о статьях (основной), "A" — планиметрия (приложение)
FILES = {"A": DATA / "corpus_A_planimetry.csv", "Aprime": DATA / "corpus_Aprime_standards.csv"}
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

## Модель: настройка

Живые ячейки шагов 1 и 5 обращаются к языковой модели через OpenAI-совместимый API. Три значения задаются здесь явно:
ключ, адрес сервера и имя модели. Ячейка проверяет, отвечает ли сервер. Если нет, ключ сбрасывается, и живые ячейки
печатают записанные ответы, остальные шаги от модели не зависят.

In [ ]:
import os

# Вариант 1. Локальная Ollama: сервер на этой машине, модель с вызовом функций (qwen-agent собрана из Modelfile
#            командой ollama create qwen-agent -f Modelfile, подойдёт и qwen2.5:3b). Ключ для Ollama любая непустая строка.
LLM_API_KEY  = "ollama"
LLM_BASE_URL = "http://127.0.0.1:11434/v1"
LLM_MODEL    = "qwen-agent"

# Вариант 2. Облачная модель: подставить свои значения вместо трёх строк выше, например
#   LLM_API_KEY = "sk-...";  LLM_BASE_URL = "https://api.openai.com/v1";  LLM_MODEL = "gpt-4o-mini"
# Вариант 3. Без модели: LLM_API_KEY = "". Живые ячейки печатают записанные ответы.

LEAN_SERVICE_URL = ""          # адрес внешнего верификатора для шага 4б. Пусто: показывается запись вердиктов

os.environ["LLM_BASE_URL"], os.environ["LLM_MODEL"] = LLM_BASE_URL, LLM_MODEL
os.environ["LEAN_SERVICE_URL"] = LEAN_SERVICE_URL
if LLM_BASE_URL.startswith(("http://127.0.0.1", "http://localhost")):
    for k in ("ALL_PROXY", "all_proxy", "HTTP_PROXY", "http_proxy", "HTTPS_PROXY", "https_proxy"):
        os.environ.pop(k, None)    # локальный сервер: системный прокси этому блокноту не нужен

# Проверка сервера. Ключ попадает в окружение только если сервер ответил.
os.environ["LLM_API_KEY"] = ""
if LLM_API_KEY:
    try:
        from openai import OpenAI
    except ModuleNotFoundError:
        OpenAI = None
        print("пакет openai не установлен в ядре блокнота: выполните %pip install openai и перезапустите ядро. Живые ячейки идут по записи")
    try:
        if OpenAI is None:
            raise RuntimeError("нет пакета openai")
        names = [m.id for m in OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL, timeout=5, max_retries=0).models.list()]
        os.environ["LLM_API_KEY"] = LLM_API_KEY
        if names and LLM_MODEL not in names and LLM_MODEL + ":latest" not in names:
            print(f"сервер отвечает, модели {LLM_MODEL} в его списке нет (есть: {', '.join(names[:6])}), вызовы могут не пройти")
        else:
            print(f"модель: {LLM_MODEL} через {LLM_BASE_URL}")
    except Exception as e:
        if OpenAI is not None:
            print(f"сервер {LLM_BASE_URL} не отвечает ({type(e).__name__}): живые ячейки идут по записи")
else:
    print("модель не задана: живые ячейки идут по записи")

## Шаг 0. Данные: журнал рецензий и корпус положений

Журнал — то, что есть у редактора на руках. Обратите внимание на четыре строки: у P2 оценка рецензента Rv3 выставлена
по шкале 5; у P3 рецензент Rv4 из организации авторов; у P5 рецензия Rv3 сдана через 27 дней; у P8 рецензия не сдана.

In [ ]:
reviews = pd.read_csv(DATA / "reviews.csv", dtype=str).fillna("")
reviews

In [ ]:
corpus = pd.read_csv(FILES[CORPUS], dtype=str).fillna("")
print(len(corpus), "положений; типы:", dict(collections.Counter(corpus["тип"])))
corpus[corpus["id"].isin(["S06", "R61", "R62", "R64", "R65", "R66"])] if CORPUS == "Aprime" else corpus.head(8)

## Шаг 1. Модель без инструментов: три ошибки

Агенту на языковой модели дали журнал (без столбца «шкала» в промпте — как это обычно и бывает) и задали вопрос редактора.
Ниже — запись характерного ответа (получена заранее; живой вызов в следующей ячейке, если модель настроена в ячейке «Модель» выше).

> **Модель:** Все восемь статей можно выносить на решение редколлегии. Средние оценки: P1 — 7,5; P2 — 6,5; P3 — 7,5;
> P4 — 8; P5 — 6,5; P6 — 4; P7 — 8,5; P8 — рецензия ожидается, но статья может быть рассмотрена по имеющимся данным.
> Требование о принятии решения опирается на п. 6.1 (две рецензии) и п. 6.4 (ответ авторов), а также на
> *ГОСТ Р 7.0.113-2021 «Рецензирование научных статей. Общие требования»*, п. 5.4.
> По статье P2: замечание рецензента Rv3 к утверждению 3 необоснованно — доказательство утверждения 3 корректно.

Три ошибки — три дефицита:

| Ошибка | Что не так | Дефицит |
|---|---|---|
| P2 — 6,5 | усреднены 4 (из 5) и 9 (из 10); 4 из 5 — это 8 из 10 | **данные без схемы**: у числа нет шкалы |
| «ГОСТ Р 7.0.113-2021» | такого стандарта нет; номер, название и пункт сочинены | **знание без источника**: форма правильного ответа без проверки |
| «все восемь можно выносить» | не проверены 6.2 (организация рецензента), 6.6 (срок), 6.5 (ответ авторов), 6.1 (две рецензии) | **проверка без правила**: правил у агента не было |
| «замечание Rv3 к утверждению 3 необоснованно, доказательство корректно» | утверждение 3 статьи P2 ложно (контрпример — равносторонний треугольник); агент оценил доказательство по виду текста | **проверка по виду**: нужен верификатор доказательств, а не чтение |

Два первых пункта про 6.1 и 6.4 — верны: они есть в положении. По тексту верное от выдуманного не отличить.

In [ ]:
import os
QUESTION = ("Какие статьи можно выносить на решение редколлегии по данным журнала рецензий, "
            "и на какие положения опирается это требование? Назови документы и пункты.")
if os.environ.get("LLM_API_KEY"):
    try:
        from openai import OpenAI
        client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ.get("LLM_BASE_URL"))
        prompt = QUESTION + "\n\nЖурнал рецензий:\n" + reviews.drop(columns=["шкала"]).to_csv(index=False)
        r = client.chat.completions.create(model=os.environ.get("LLM_MODEL", "gpt-4o-mini"),
                                           messages=[{"role": "system", "content": "Отвечай по-русски."},
                                                     {"role": "user", "content": prompt}])
        print(r.choices[0].message.content)
    except Exception as e:
        print("вызов не удался:", e)
else:
    print("модель не настроена или не отвечает: используем запись ответа выше.")

## Шаг 2. Поиск по словам (BM25)

Вторая половина вопроса редактора — «на что опирается требование 6.5» — это вопрос типа (2), о зависимости.
Попробуем ответить на него полнотекстовым поиском по корпусу положений. BM25 (Robertson, Zaragoza 2009) — стандартная
функция ранжирования по словам: частота слова в документе, редкость слова в корпусе, поправка на длину. Реализация — 25 строк.

In [ ]:
def tokenize(text):
    return re.findall(r"[а-яёa-z0-9]+", text.lower())

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = [tokenize(d) for d in docs]
        self.k1, self.b = k1, b
        self.N = len(self.docs)
        self.avgdl = sum(len(d) for d in self.docs) / self.N
        self.df = collections.Counter(w for d in self.docs for w in set(d))
        self.tf = [collections.Counter(d) for d in self.docs]

    def idf(self, w):
        n = self.df.get(w, 0)
        return math.log((self.N - n + 0.5) / (n + 0.5) + 1)

    def score(self, query, i):
        s, dl = 0.0, len(self.docs[i])
        for w in tokenize(query):
            f = self.tf[i].get(w, 0)
            if f:
                s += self.idf(w) * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * dl / self.avgdl))
        return s

    def search(self, query, k=5):
        ranked = sorted(range(self.N), key=lambda i: -self.score(query, i))
        return [(i, self.score(query, i)) for i in ranked[:k] if self.score(query, i) > 0]

texts = (corpus["название"] + ". " + corpus["формулировка"]).tolist()
bm25 = BM25(texts)

QUERY = {"Aprime": "на какие положения опирается требование 6.5 о принятии решения редколлегией",
         "A":      "на какие результаты опирается теорема Пифагора"}[CORPUS]
print("Запрос:", QUERY)
for i, s in bm25.search(QUERY, k=5):
    print(f"{s:5.2f}  {corpus.loc[i,'id']:4}  {corpus.loc[i,'название']}")

Поиск нашёл **сам пункт 6.5** (в нём есть слова запроса) и документы с похожими словами. Цепочку «на что он опирается»
он не нашёл и не мог найти: в тексте 6.5 нет слов «раздел 3», «термин Рецензент» или «требование 6.3».
Ответ на вопрос о зависимости лежит не в тексте документа, а в связях между документами.

## Шаг 3. Граф знаний: транзитивный путь

Те же строки корпуса, но поле `зависит_от` становится рёбрами графа: тройки (положение, зависит_от, положение).
Схема минимальная — класс на тип и одно транзитивное свойство. На Л2 она станет онтологией в OWL, на Л4 — полноценным RDF-графом.

In [ ]:
from rdflib import Graph, Namespace, Literal, RDF, RDFS

EX = Namespace("http://example.org/kg#")
g = Graph()
g.bind("ex", EX)

def uri(i):
    return EX[i]

for _, row in corpus.iterrows():
    s = uri(row["id"])
    g.add((s, RDF.type, EX[row["тип"]]))
    g.add((s, RDFS.label, Literal(row["название"], lang="ru")))
    g.add((s, EX.formulation, Literal(row["формулировка"], lang="ru")))
    g.add((s, EX.source, Literal(row["источник"])))
    for dep in filter(None, row["зависит_от"].split(";")):
        g.add((s, EX.dependsOn, uri(dep.strip())))

print(len(g), "троек;", len(list(g.subjects(RDF.type, None))), "узлов;",
      len(list(g.triples((None, EX.dependsOn, None)))), "рёбер зависит_от")

In [ ]:
START = {"Aprime": "R65", "A": "A14"}[CORPUS]
title = corpus.set_index("id")

q = f"""
PREFIX ex: <http://example.org/kg#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?dep ?type ?label WHERE {{
  ex:{START} ex:dependsOn+ ?dep .
  ?dep a ?type ; rdfs:label ?label .
}}
"""
rows = [(str(r.dep).split("#")[1], str(r.type).split("#")[1], str(r.label)) for r in g.query(q)]
print(f"На что транзитивно опирается {START} «{title.loc[START,'название']}»: {len(rows)} положений")
deps = pd.DataFrame(rows, columns=["id", "тип", "название"]).sort_values(["тип", "id"]).reset_index(drop=True)
deps

In [ ]:
def chains(start, depth=0, seen=None, out=None):
    seen = seen if seen is not None else set(); out = out if out is not None else []
    for dep in sorted(g.objects(uri(start), EX.dependsOn)):
        d = str(dep).split("#")[1]
        out.append("  " * depth + f"{start} → {d}  ({title.loc[d,'название']})")
        if d not in seen:
            seen.add(d)
            chains(d, depth + 1, seen, out)
    return out

print("\n".join(chains(START)))

Знак `+` после свойства в SPARQL — «один или больше шагов по этому ребру»: транзитивное замыкание. Ответ — не список
похожих документов, а **путь**, каждый шаг которого записан в корпусе с источником. Это и есть ответ на вторую половину
вопроса редактора: 6.5 опирается на 6.1 и 6.4, те — на 6.3 и термины, термины — на раздел 3, раздел 3 — на раздел 1.

## Шаг 4. Проверка по данным: правило + журнал

Первая половина вопроса — «какие статьи можно выносить» — это вопрос типа (4), о проверке. Его не решает ни один канал поиска:
нужны **правила** (из положения) и **данные** (журнал). Правила — четыре, каждое в одну строку. Сначала — схема данных:
оценки приводятся к одной шкале, даты — к числам дней.

In [ ]:
rv = reviews.copy()
rv["сдана"] = rv["сдано"] != ""
rv["оценка_10"] = pd.to_numeric(rv["оценка"], errors="coerce") * 10 / pd.to_numeric(rv["шкала"], errors="coerce")
rv["дней"] = (pd.to_datetime(rv["сдано"], errors="coerce") - pd.to_datetime(rv["назначено"])).dt.days
rv["своя_организация"] = rv["рецензент_организация"] == rv["авторы_организация"]
rv["просрочена"] = rv["дней"] > 21
rv[["статья", "рецензент", "оценка", "шкала", "оценка_10", "дней", "своя_организация", "просрочена", "ответ_авторов"]]

In [ ]:
RULES = {
    "6.1": ("не менее двух сданных рецензий",           lambda d: d["сдана"].sum() >= 2),
    "6.2": ("рецензент не из организации авторов",      lambda d: not d.loc[d["сдана"], "своя_организация"].any()),
    "6.6": ("рецензии сданы в 21 день",                 lambda d: not d.loc[d["сдана"], "просрочена"].any()),
    "6.5": ("есть ответ авторов на замечания",          lambda d: d["сдана"].sum() >= 2 and (d.loc[d["сдана"], "ответ_авторов"] == "да").all()),
}

verdict = []
for paper, d in rv.groupby("статья", sort=True):
    failed = [f"{k} ({RULES[k][0]})" for k in RULES if not RULES[k][1](d)]
    mean10 = d.loc[d["сдана"], "оценка_10"].mean()
    verdict.append({"статья": paper, "выносить": "да" if not failed else "нет",
                    "средняя оценка (шкала 10)": round(mean10, 1) if pd.notna(mean10) else None,
                    "причина": "; ".join(failed) or "—"})
verdict = pd.DataFrame(verdict)
verdict

Три статьи готовы к решению — P1, P2, P7; пять — нет, и у каждой указан пункт положения. Заметьте P2: после приведения
шкалы её средняя оценка 8,5, а не 6,5 — «слабая статья» из ответа модели была ошибкой схемы, а не рецензентов.

Проверка заняла четыре правила и пятнадцать строк кода. Но написать их можно только зная положение — то есть имея
структуру знания, а не только текст.

## Шаг 4б. Утверждения статьи P2: проверка доказательств внешним сервисом

Статья P2 — по математике («О медианах и высотах прямоугольного треугольника»). В ней три утверждения; рецензент Rv3 сделал
замечание к утверждению 3. Проверить, кто прав — рецензент или авторы, — правилом над журналом нельзя: это вопрос о **выводе**,
а не о данных. Инструмент для него — верификатор доказательств. В стенде это Lean 4 как сервис (`lean_check` в MCP-сервере
агента AI4Science); контракт: формальная постановка + доказательство → «принято» или ошибка.

Ячейка ниже шлёт постановки в сервис, если задан `LEAN_SERVICE_URL`; без адреса показывает запись вердиктов из
`data/paper_P2_claims.csv` (формализации сделаны в координатах: прямой угол в начале координат, катеты a и b).

In [ ]:
claims = pd.read_csv(DATA / "paper_P2_claims.csv", dtype=str).fillna("")
claims[["id", "вид", "формулировка", "опирается_на", "замечание_рецензента"]]

In [ ]:
import os, json, urllib.request

def lean_check(statement: str):
    """Проверка через внешний сервис (Lean 4 + Mathlib). Без адреса — None, используем запись вердикта."""
    url = os.environ.get("LEAN_SERVICE_URL")
    if not url:
        return None
    req = urllib.request.Request(url, data=json.dumps({"code": statement}).encode(),
                                 headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.load(resp).get("verdict", "?")

rows = []
for _, c in claims.iterrows():
    live = lean_check(c["формальная_постановка"])
    rows.append({"id": c["id"], "формальная постановка": c["формальная_постановка"][:70] + "…",
                 "вердикт": live or c["вердикт_lean"] + ("" if live else "  [запись]")})
lean_verdicts = pd.DataFrame(rows)
lean_verdicts

Итог: C1 и C2 приняты, C3 отклонено — **рецензент прав, авторы в ответе исправили условие**. Агент, судивший по виду текста,
ошибся; верификатор, судящий по выводу, — нет.

Три оговорки, без которых этот шаг вводит в заблуждение:

- верификатор проверяет доказательство **относительно записанной постановки**; совпадает ли постановка с утверждением статьи —
  проверяет человек (формализация в координатах — уже выбор);
- заглушка `sorry` делает «доказательство» принятым формально; надёжная проверка смотрит на использованные аксиомы;
- формализуемо не всё: «сдано в 21 день» — не теорема, а факт о данных, для него верификатор — правило над журналом (шаг 4).
  Два инструмента под два вида утверждений.

Это место курса, где математика встречается с агентами: доказатели на языковых моделях, автоформализация, агент-проверщик
с подсказками из графа — Модуль 4 и Л19–Л20. Математика здесь — пример области с бинарным верификатором; в вашей области
им будет тест, проверка данных или воспроизведённый эксперимент.

## Шаг 5. Агент собирает ответ сам

Шаги 2–4б мы запускали руками. Агент — это цикл: модель пишет вызов инструмента, приложение исполняет его,
результат возвращается в контекст, и так до ответа без вызова. Инструментов четыре — те же функции, что выше:
`search_words` (шаг 2), `graph_deps` (шаг 3), `check_rules` (шаг 4), `lean_check_claims` (шаг 4б). Модель сама решает,
что и в каком порядке вызвать; ни один вердикт она не порождает — все приходят из инструментов.

Ячейка работает с любой моделью, поддерживающей вызов функций через OpenAI-совместимый API (`LLM_API_KEY`,
`LLM_BASE_URL`, `LLM_MODEL`), настроенной в ячейке «Модель» в начале блокнота. Если модель не настроена или сервер не отвечает, печатается **образец протокола** такого прогона, составленный по выводам
инструментов из шагов 2–4б и помеченный `[запись]`.

In [ ]:
import json, os

def search_words(query: str, k: int = 5):
    return [{"id": corpus.loc[i, "id"], "название": corpus.loc[i, "название"], "bm25": round(s, 2)}
            for i, s in bm25.search(query, k=k)]

def graph_deps(doc_id: str):
    qd = ("PREFIX ex: <http://example.org/kg#> "
          f"SELECT ?dep WHERE {{ ex:{doc_id} ex:dependsOn+ ?dep . }}")
    ids = sorted(str(r.dep).split("#")[1] for r in g.query(qd))
    return {"опирается_на": [f"{i} {title.loc[i, 'название']}" for i in ids],
            "цепочки": chains(doc_id)[:12]}

def check_rules():
    return verdict.to_dict(orient="records")

def lean_check_claims():
    return lean_verdicts.to_dict(orient="records")

TOOLS = [
  {"type": "function", "function": {"name": "search_words", "description": "Поиск по словам (BM25) по корпусу положения.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
  {"type": "function", "function": {"name": "graph_deps", "description": "На какие положения транзитивно опирается документ (граф знаний, зависит_от+).",
     "parameters": {"type": "object", "properties": {"doc_id": {"type": "string", "description": "например R65"}}, "required": ["doc_id"]}}},
  {"type": "function", "function": {"name": "check_rules", "description": "Применить правила 6.1, 6.2, 6.5, 6.6 положения к журналу рецензий; вердикт по каждой статье.",
     "parameters": {"type": "object", "properties": {}}}},
  {"type": "function", "function": {"name": "lean_check_claims", "description": "Вердикты внешнего верификатора доказательств по утверждениям статьи P2.",
     "parameters": {"type": "object", "properties": {}}}},
]
IMPL = {"search_words": search_words, "graph_deps": graph_deps, "check_rules": check_rules, "lean_check_claims": lean_check_claims}

SYSTEM = ("Отвечай по-русски. Ты ассистент редактора сборника. Отвечай только по результатам инструментов. "
          "Каждую часть ответа помечай: [проверено: инструмент] или [найдено: инструмент]. "
          "Документы и пункты, которых нет в результатах инструментов, не называй; если что-то не удалось подтвердить — пиши [не подтверждено]. "
          "Идентификатор требования 6.5 в корпусе — R65.")

def run_agent(question, max_steps=8):
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ.get("LLM_BASE_URL"))
    model = os.environ.get("LLM_MODEL", "gpt-4o-mini")
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=model, messages=messages, tools=TOOLS)
        m = r.choices[0].message
        messages.append(m)
        if not m.tool_calls:
            print(f"\n[{step}] ответ модели:\n{m.content}")
            return m.content
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            result = IMPL[tc.function.name](**args)
            text = json.dumps(result, ensure_ascii=False)
            print(f"[{step}] модель вызвала {tc.function.name}({args})")
            print(f"    ← {text[:160]}{'…' if len(text) > 160 else ''}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": text})
    print("лимит шагов исчерпан")

RECORD = """[запись] образец протокола прогона — без ключа API
[1] модель вызвала check_rules({})
    ← [{"статья": "P1", "выносить": "да", "средняя оценка (шкала 10)": 7.5, "причина": "—"}, {"статья": "P2", "выносить": "да", "средняя оценка (шкала 10)": 8.5, …
[2] модель вызвала search_words({'query': 'принятие решения редколлегией требование'})
    ← [{"id": "R65", "название": "6.5 Принятие решения", "bm25": 5.14}, {"id": "R91", "название": "9.1 Повторная подача", "bm25": 2.34}]
[3] модель вызвала graph_deps({'doc_id': 'R65'})
    ← {"опирается_на": ["R61 6.1 Число рецензий", "R63 6.3 Содержание рецензии", "R64 6.4 Ответ на замечания", "S01 Раздел 1. Область применения", …
[4] модель вызвала lean_check_claims({})
    ← [{"id": "C1", …, "вердикт": "принято  [запись]"}, {"id": "C2", …, "вердикт": "принято  [запись]"}, {"id": "C3", …, "вердикт": "отклонено: доказательство не собирается; контрпример  [запись]"}]

[5] ответ модели:
Можно выносить на решение: P1, P2, P7 [проверено: check_rules]. Нельзя: P3 — 6.2 (рецензент из организации авторов);
P4 — 6.1 (одна рецензия); P5 — 6.6 (просрочена); P6 — 6.5 (нет ответа авторов); P8 — 6.1 (рецензия не сдана) [проверено: check_rules].
Требование 6.5 (R65) опирается на 6.1, 6.4, 6.3, термины Решение, Ответ авторов, Замечание, Рецензия, Рецензент,
Редколлегия, Автор, Статья и разделы 6, 3, 1 [найдено: graph_deps]. По статье P2: утверждения C1, C2 приняты,
C3 отклонено — замечание рецензента обосновано [проверено: lean_check_claims]. Иных нормативных документов
(стандартов) в результатах инструментов нет [не подтверждено]."""

if os.environ.get("LLM_API_KEY"):
    run_agent(QUESTION)
elif CORPUS == "Aprime":
    print(RECORD)
else:
    print("запись протокола относится к вопросу редактора (CORPUS = 'Aprime'); для живого прогона задайте LLM_API_KEY")

Смотреть на две вещи. Порядок вызовов выбрала модель, и в живом прогоне он может отличаться от записи — это нормально,
важно, что каждый факт в ответе пришёл из инструмента. И второе: пункты, которых в результатах инструментов нет,
модель по инструкции не называет — несуществующему ГОСТу здесь взяться неоткуда. Агент — не пятый слой, а оркестратор
четырёх; сам он ничего не проверяет.

## Шаг 6. Сборка ответа: найдено, проверено, не подтверждено, путь

Так выглядит ответ ассистента, который курс учит строить: каждая часть помечена, откуда она и чем проверена.

In [ ]:
ok = verdict[verdict["выносить"] == "да"]["статья"].tolist()
no = verdict[verdict["выносить"] == "нет"]
chain = " → ".join(["R65", "R64", "R63", "T07", "T06", "S03", "S01"])

print("ОТВЕТ РЕДАКТОРУ\n")
print("Проверено по журналу и правилам 6.1, 6.2, 6.5, 6.6 [источник: reviews.csv; corpus R61, R62, R65, R66]:")
print("  можно выносить на решение:", ", ".join(ok))
for _, r in no.iterrows():
    print(f"  нельзя: {r['статья']} — {r['причина']}")
print("\nНайдено по графу [источник: corpus, зависит_от+]:")
print(f"  требование 6.5 транзитивно опирается на {len(deps)} положений; путь до базового раздела: {chain}")
print("\nПроверено верификатором доказательств [источник: paper_P2_claims.csv; lean_check]:")
print("  P2: C1, C2 — приняты; C3 — отклонено (контрпример). Замечание рецензента Rv3 обосновано, ответ авторов принят.")
print("\nНе подтверждено:")
print("  ссылка на «ГОСТ Р 7.0.113-2021» из ответа модели — в корпусе и во внешних источниках не найдена; помечена [порождено, не проверено]")

## Шаг 7. Что увидели

| Тип вопроса | Модель без инструментов | Слова (BM25) | Граф + SPARQL | Правило + данные |
|---|---|---|---|---|
| (1) факт: «что требует 6.5» | верно, но без источника | находит | находит (по метке) | — |
| (2) зависимость: «на что опирается 6.5» | частично верно + выдуманный ГОСТ | только сам документ | вся цепочка, 14 положений | — |
| (3) смысл: «есть ли норма о сроках в других словах» | правдоподобно | частично | через связанные сущности | — |
| (4) проверка: «какие статьи можно выносить» | «все восемь» — неверно | нет | нет | P1, P2, P7 с причинами |
| (4б) проверка доказательства: «верно ли утверждение 3 статьи P2» | «корректно» — неверно | нет | нет | нет; **верификатор доказательств** (Lean-сервис): отклонено, рецензент прав |

Разные слои закрывают разные вопросы. Модуль 1 строит граф, Модуль 2 — каналы поиска и их гибрид, Модуль 3 — агента,
который выбирает слой под вопрос, Часть II — проверку и объяснение.

## По желанию: тот же запрос к большому графу

Стенд — граф SciLib (~66 млн троек, 724 класса OWL) с открытой точкой доступа SPARQL. Ячейка шлёт туда запрос той же формы;
адрес и имена свойств задаются в переменных. Без адреса ячейка пропускается.

In [ ]:
SCILIB_ENDPOINT = None   # например "https://.../sparql"
SCILIB_QUERY = """
SELECT ?dep ?label WHERE { <URI_УТВЕРЖДЕНИЯ> <URI_СВОЙСТВА_ЗАВИСИТ_ОТ>+ ?dep . ?dep rdfs:label ?label } LIMIT 50
"""
if SCILIB_ENDPOINT:
    import urllib.request, urllib.parse, json
    url = SCILIB_ENDPOINT + "?" + urllib.parse.urlencode({"query": SCILIB_QUERY})
    req = urllib.request.Request(url, headers={"Accept": "application/sparql-results+json"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        res = json.load(resp)
    for b in res["results"]["bindings"]:
        print(b["dep"]["value"], "—", b["label"]["value"])
else:
    print("SCILIB_ENDPOINT не задан — показываем только учебный граф.")

## Что повторить дома (часть 1 ДЗ)

1. Запустить блокнот целиком; затем с `CORPUS = "A"` — шаги 2–3 на корпусе планиметрии.
2. Заменить корпус на **свой** (≥ 30 утверждений своей области в том же формате) и задать свои вопросы типов (1) и (2).
3. Если в вашей области есть проверяемый факт с данными — записать одно правило по образцу шага 4.
4. Убедиться, что для хотя бы одного вопроса о зависимости BM25 находит документ, а граф — цепочку.
5. Записать в `diary.md`, что получилось и что не получилось, с пометками [написано] / [порождено] / [источник].